# Step 06 — Assembly: the LLM answers on the buildings

Step 05 asked the model, once per call, what happens inside each building and
what kind of building it is. This notebook puts those answers on the building
polygons, so that **every building carries its classes**: the class_only
buildings that shared a call get their signature's answer, `work` is added by
rule, and the result is written with the POI pairs the redistribution needs.
No model call is made here.

| | |
|---|---|
| **Reads** | `data/output/04_buildings_enriched.gpkg` — layers `buildings` (the slim column set of `config.ASSEMBLY_BUILDING_COLS`) and `building_pois`; `05_llm_plan.parquet` — who was answered by which call; `05_llm_answers.jsonl` — the full run from the Linux machine |
| **Writes** | `data/output/06_buildings_classified.gpkg` — layer `buildings` (one per building, with its answer) and `building_pois` (step 04's, unchanged) |
| **Needs** | `pyogrio`, `geopandas`, `pandas`, `pyarrow` |

## State of this notebook

| step | | status |
|---|---|---|
| **06.1** | **The inputs** — the buildings, the plan and the valid answers, checked against each other | **implemented** |
| **06.2** | **The answers on the buildings** — each building gets the answer of the call that answered it; signature members get their representative's, marked as copied | **implemented** |
| **06.3** | **The work rule** — `work` wherever any other label is present, `work_from` saying who put it there | **implemented** |
| **06.4** | **The result and the file** — what the buildings now carry, written with the POI pairs | **implemented** |
| **06.5** | **Validation** — against the rule baseline in `activities` and the annotated Bosserhof set of the previous pipeline | pending |

In [1]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError('Cannot find the pipeline root (the folder containing config.py). '
                       f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.')
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import time
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio

from config import (
    TARGET_CRS, OUTPUT_DIR, ENRICHED_BUILDINGS_FILE,
    LLM_PLAN_FILE, LLM_ANSWERS_FILE, LLM_ACTIVITY_LABELS, WORK_IMPLIED_BY,
    LLM_BOSSERHOF_HEADLINES, LLM_CONFIDENCE_LEVELS,
    ASSEMBLY_BUILDING_COLS, CLASSIFIED_BUILDINGS_FILE,
)
from lib.checks import require_file, require_non_empty, require_unique, require_cols, require_crs
from lib.llm_routing import PLAN_COLUMNS
from lib.llm_client import prompt_sha
from lib.llm_run import load_answers, valid_answers, apply_work_rule

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 90)

print('Root  :', ROOT_DIR)
print('Input :', ENRICHED_BUILDINGS_FILE.name, '+', LLM_PLAN_FILE.name, '+', LLM_ANSWERS_FILE.name)
print('Output:', CLASSIFIED_BUILDINGS_FILE.name)

Root  : C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
Input : 04_buildings_enriched.gpkg + 05_llm_plan.parquet + 05_llm_answers.jsonl
Output: 06_buildings_classified.gpkg


## 1. The inputs

Three files that must agree with each other:

* the **buildings** of step 04 — every one of them was planned in step 05, so
  the plan and the layer must hold exactly the same ids;
* the **plan** — for every building, the call that answered it
  (`answered_by`): the building itself (`route = building`), or the median-area
  member of its signature (`route = signature`);
* the **answers** of the full run, which came from the Linux machine. Only
  answers that are valid under the current prompt count: `valid_answers` keeps
  the latest valid line per call and ignores failed lines and answers given
  under another prompt. If `prompt_sha()` differs from the one the run used,
  nothing counts, and `config.LLM_SYSTEM_PROMPT` has to be synced first.

A call without a valid answer stops the notebook with the ids to re-ask on the
Linux machine (`python scripts/05_run_llm.py --ids ...`).

In [2]:
require_file(ENRICHED_BUILDINGS_FILE, 'enriched buildings (step 04)')
require_file(LLM_PLAN_FILE, 'routing plan (step 05.4)')
require_file(LLM_ANSWERS_FILE, 'answers of the full run (step 05.5)')
print()

# --- the buildings: the slim column set and the geometry -----------------------
_fields = set(pyogrio.read_info(ENRICHED_BUILDINGS_FILE, layer='buildings')['fields'])
_absent = [c for c in ASSEMBLY_BUILDING_COLS if c not in _fields]
if _absent:
    raise AssertionError(f'config.ASSEMBLY_BUILDING_COLS names columns the step 04 layer does not have: {_absent}')
print(f'Reading the step 04 buildings ({len(ASSEMBLY_BUILDING_COLS)} of {len(_fields)} columns) ...', flush=True)
t0 = time.perf_counter()
bld = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='buildings', columns=list(ASSEMBLY_BUILDING_COLS))
bld = bld[list(ASSEMBLY_BUILDING_COLS) + ['geometry']]
print(f'  ok  {len(bld):,} buildings  [{time.perf_counter() - t0:,.1f}s]')
require_non_empty(bld, 'buildings')
require_unique(bld, 'building_id', 'buildings')
require_crs(bld, TARGET_CRS, 'buildings')

# --- the POI pairs, carried over unchanged ---------------------------------------
pairs = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='building_pois')
require_cols(pairs, ['building_id', 'poi_id', 'share_in_building', 'share_of_site'], 'building_pois')
_orphans = set(pairs['building_id']) - set(bld['building_id'])
if _orphans:
    raise AssertionError(f'{len(_orphans):,} building_pois rows point at buildings not in the layer, e.g. {sorted(_orphans)[:3]}')
print(f'  ok  {len(pairs):,} building-POI pairs on {pairs["building_id"].nunique():,} buildings')

# --- the plan ----------------------------------------------------------------------
print()
plan = pd.read_parquet(LLM_PLAN_FILE)
require_cols(plan, PLAN_COLUMNS, 'plan')
require_unique(plan, 'building_id', 'plan')
_not_planned = set(bld['building_id']) - set(plan['building_id'])
_not_built = set(plan['building_id']) - set(bld['building_id'])
if _not_planned or _not_built:
    raise AssertionError(f'plan and buildings disagree: {len(_not_planned):,} buildings not in the plan '
                         f'(e.g. {sorted(_not_planned)[:3]}), {len(_not_built):,} planned buildings not in the layer '
                         f'(e.g. {sorted(_not_built)[:3]}) - step 04 changed after the plan was made')
print(f'  ok  the plan holds exactly the {len(bld):,} buildings of the layer; '
      f"{plan['answered_by'].nunique():,} calls, {plan['route'].value_counts().to_dict()}")

# --- the answers ---------------------------------------------------------------------
print()
sha = prompt_sha()
raw = load_answers(LLM_ANSWERS_FILE)
answers = valid_answers(raw, sha)
_failed = int((raw['ok'] != True).sum())                                             # noqa: E712 - object column
_other = int(((raw['ok'] == True) & (raw['prompt_sha'] != sha)).sum())               # noqa: E712
print(f'  ..  {LLM_ANSWERS_FILE.name}: {len(raw):,} lines - {len(answers):,} valid under prompt {sha}, '
      f'{_failed:,} failed line(s), {_other:,} answer(s) under another prompt')
if answers.empty:
    raise AssertionError(f'no valid answer under prompt {sha}: config.LLM_SYSTEM_PROMPT differs from the one the run used')
print(f"  ..  model(s): {answers['model'].value_counts().to_dict()}; answered {answers['ts'].min()} to {answers['ts'].max()}")

_unanswered = sorted(set(plan['answered_by']) - set(answers['building_id']))
if _unanswered:
    raise AssertionError(f'{len(_unanswered):,} call(s) have no valid answer. Re-ask them on the Linux machine:\n'
                         f"  python scripts/05_run_llm.py --ids {','.join(_unanswered)}")
_unused = set(answers['building_id']) - set(plan['answered_by'])
print(f"  ok  every one of the {plan['answered_by'].nunique():,} calls has a valid answer"
      + (f'; {len(_unused):,} answer(s) for buildings the plan does not ask are ignored' if _unused else ''))

  ok  04_buildings_enriched.gpkg (42.0 MB)
  ok  05_llm_plan.parquet (0.7 MB)
  ok  05_llm_answers.jsonl (25.1 MB)

Reading the step 04 buildings (15 of 43 columns) ...


  ok  39,786 buildings  [0.7s]
  ok  buildings: 39,786 rows
  ok  buildings.building_id: unique and non-null (39,786)
  ok  buildings: CRS EPSG:25832


  ok  building_pois: has ['building_id', 'poi_id', 'share_in_building', 'share_of_site']


  ok  41,285 building-POI pairs on 30,103 buildings



  ok  plan: has ['building_id', 'evidence', 'route', 'signature', 'answered_by', 'n_in_signature']
  ok  plan.building_id: unique and non-null (39,786)


  ok  the plan holds exactly the 39,786 buildings of the layer; 34,993 calls, {'building': 33734, 'signature': 6052}



  ..  05_llm_answers.jsonl: 34,994 lines - 34,993 valid under prompt 795433ed9feb, 1 failed line(s), 0 answer(s) under another prompt
  ..  model(s): {'gpt-oss-120b': 34993}; answered 2026-09-15T11:44:30 to 2026-09-23T09:25:02


  ok  every one of the 34,993 calls has a valid answer


## 2. The answers on the buildings

Every building takes the answer of the call that answered it:
`plan.answered_by → answers.building_id`. For the 33,734 buildings with their
own call that is their own answer. The 6,052 class_only buildings were grouped
into signatures — same register class, footprint type, land use, area band and
height band — and each signature was asked once, on the record of its
median-area member. Every member now gets that answer.

`answer_copied` marks the buildings whose answer was given for another
building's record: all signature members except the representative itself. It
keeps them apart from an answer a building got on its own, which matters for
the validation — a copied answer is exactly as good as the signature is
homogeneous.

In [3]:
_ANSWER_COLS = ['interpreted_type', 'mid_labels', 'bosserhof_class', 'confidence', 'reason']
cls = plan.merge(answers.set_index('building_id')[_ANSWER_COLS], left_on='answered_by', right_index=True,
                 how='left', validate='many_to_one')
cls['answer_copied'] = cls['building_id'] != cls['answered_by']

# --- contract: every building has a complete answer ------------------------------
_holes = cls[_ANSWER_COLS].isna().any(axis=1) | (cls['mid_labels'].map(len) == 0)
if _holes.any():
    raise AssertionError(f'{int(_holes.sum()):,} buildings without a complete answer, e.g. '
                         f"{cls.loc[_holes, 'building_id'].head(3).tolist()}")
if (cls['answer_copied'] & (cls['route'] != 'signature')).any():
    raise AssertionError('a building with its own call carries a copied answer')
_sig = cls['route'] == 'signature'
_spread = cls[_sig].groupby('signature')[['interpreted_type', 'bosserhof_class', 'confidence']].nunique()
if (_spread > 1).any().any():
    raise AssertionError('the members of a signature carry different answers')
print(f'  ok  all {len(cls):,} buildings carry a complete answer')
print(f"  ..  {int((~_sig).sum()):,} answered on their own record")
print(f"  ..  {int(_sig.sum()):,} class_only buildings in {cls.loc[_sig, 'signature'].nunique():,} signatures: "
      f"{int((_sig & ~cls['answer_copied']).sum()):,} representatives answered on their own record, "
      f"{int(cls['answer_copied'].sum()):,} members with the representative's answer copied")

print()
print('  ..  the eight largest signatures and the answer their members now share:')
_top = (cls[_sig].groupby('signature')
        .agg(buildings=('building_id', 'size'), answered_by=('answered_by', 'first'),
             labels=('mid_labels', lambda s: ', '.join(s.iloc[0])), bosserhof_class=('bosserhof_class', 'first'),
             confidence=('confidence', 'first'))
        .sort_values('buildings', ascending=False).head(8))
with pd.option_context('display.max_colwidth', 110):
    print(_top.to_string())

  ok  all 39,786 buildings carry a complete answer
  ..  33,734 answered on their own record
  ..  6,052 class_only buildings in 1,259 signatures: 1,259 representatives answered on their own record, 4,793 members with the representative's answer copied

  ..  the eight largest signatures and the answer their members now share:


                                                                                                                                    buildings       answered_by                   labels                     bosserhof_class confidence
signature                                                                                                                                                                                                                              
Residential buildings with trade and services | footprint yes | land commercial services | - | osm residential | <250 m2 | 10-20 m        359  DENIAL0600001y9f  work, errands, business                            Services     medium
Residential buildings with trade and services | footprint yes | land commercial services | - | osm residential | <250 m2 | 5-10 m         219  DENIAL03000048Jp            work, errands          customer-oriented services     medium
Buildings for trade and services | footprint yes | land commercial servi

## 3. The work rule

`work` is defined in the prompt as the MiD trip purpose: people come because
they are employed here. The staff of a shop, a school or a gym are a fact, not a
judgement, so the model is not asked to decide it for every building — the rule
adds it. `WORK_IMPLIED_BY` in config is every activity label except `work`;
wherever one of them is present, the building also gets `work`
(`lib/llm_run.apply_work_rule`).

Two label columns, so nothing the model said is lost:

* `llm_labels` — the model's labels, as it gave them;
* `mid_labels` — what the later steps use: the model's labels plus `work` by
  rule, in the order of `LLM_ACTIVITY_LABELS`, so the same set is always the
  same string.

`work_from` says who put `work` there: `llm` (the model, and no other label
implies it — work is the only label), `rule` (only the rule) or `both` (the
model gave it next to a label that implies it). Every building ends up with
`work`: each has at least one label, and every label is either `work` or
implies it.

In [4]:
_order = {l: i for i, l in enumerate(LLM_ACTIVITY_LABELS)}
_raw = cls['mid_labels'].map(list)
_dup = int((_raw.map(len) != _raw.map(lambda ls: len(set(ls)))).sum())
_final = _raw.map(lambda ls: sorted(set(apply_work_rule(ls)), key=_order.__getitem__))

_by_llm = _raw.map(lambda ls: 'work' in ls)
_implied = _raw.map(lambda ls: any(l in WORK_IMPLIED_BY for l in ls))
cls['work_from'] = np.select([_by_llm & _implied, _by_llm, _implied], ['both', 'llm', 'rule'], default='')
if (cls['work_from'] == '').any() or not _final.map(lambda ls: 'work' in ls).all():
    raise AssertionError('a building without work after the rule')

cls['llm_labels'] = _raw.map(';'.join)
cls['mid_labels'] = _final.map(';'.join)

_wf = cls['work_from'].value_counts()
print(f"  ok  work on all {len(cls):,} buildings after the rule")
print(f"  ..  work_from: llm {_wf.get('llm', 0):,} (work is the only label) · both {_wf.get('both', 0):,} · "
      f"rule {_wf.get('rule', 0):,} (the rule added it)")
print(f'  ..  {_dup:,} answer(s) listed a label twice; the set is kept once in mid_labels')

  ok  work on all 39,786 buildings after the rule
  ..  work_from: llm 12,161 (work is the only label) · both 13,952 · rule 13,673 (the rule added it)
  ..  0 answer(s) listed a label twice; the set is kept once in mid_labels


## 4. The result and the file

What the buildings carry now — counted over **buildings**, so a signature
answer counts once for each member, not once per call. The comparison with
the rule baseline in `activities` is 06.5.

In [5]:
_n = len(cls)

def _share(counts):
    return ' · '.join(f'{k} {v:,} ({100 * v / _n:.1f} %)' for k, v in counts.items())

print('confidence:', _share(cls['confidence'].value_counts().reindex(list(LLM_CONFIDENCE_LEVELS), fill_value=0)))
print()
print('confidence by evidence group (buildings):')
print(pd.crosstab(cls['evidence'], cls['confidence'], margins=True)[list(LLM_CONFIDENCE_LEVELS) + ['All']].to_string())
print()

_lab = pd.Series([l for ls in _final for l in ls]).value_counts().reindex(list(LLM_ACTIVITY_LABELS), fill_value=0)
print('buildings per label, after the work rule:')
print('  ' + '\n  '.join(f'{k:<17} {v:>7,}  {100 * v / _n:5.1f} %' for k, v in _lab.items()))
print()
print('labels per building (with work):', _final.map(len).value_counts().sort_index().to_dict())
print()
print('the twelve most frequent label sets:')
print(cls['mid_labels'].value_counts().head(12).to_string())
print()

_head = cls['bosserhof_class'].isin(LLM_BOSSERHOF_HEADLINES)
print(f'Bosserhof: {int(_head.sum()):,} buildings with a headline only, {int((~_head).sum()):,} with a subcategory')
print(cls['bosserhof_class'].value_counts().head(15).to_string())

confidence: high 20,073 (50.5 %) · medium 18,418 (46.3 %) · low 1,295 (3.3 %)

confidence by evidence group (buildings):
confidence    high  medium   low    All
evidence                               
class_only      72    5829   151   6052
name_only      554     891    22   1467
poi_or_site  19387    9682  1034  30103
tag_only        60    2016    88   2164
All          20073   18418  1295  39786

buildings per label, after the work rule:
  work               39,786  100.0 %
  university            341    0.9 %
  school              1,385    3.5 %
  childcare             864    2.2 %
  retail_daily        3,454    8.7 %
  retail_non_daily    4,383   11.0 %
  leisure             4,926   12.4 %
  sports              1,877    4.7 %
  errands            10,203   25.6 %
  meetup              3,625    9.1 %
  lessons               772    1.9 %
  business            4,173   10.5 %

labels per building (with work): {1: 12161, 2: 20601, 3: 5974, 4: 770, 5: 261, 6: 14, 7: 5}

the twelve most fr

In [6]:
_KEEP_PLAN = ['evidence', 'route', 'signature', 'answered_by', 'n_in_signature', 'answer_copied']
_KEEP_LLM = ['llm_type', 'llm_labels', 'mid_labels', 'work_from', 'bosserhof_class', 'llm_confidence', 'llm_reason']
_cls = cls.rename(columns={'interpreted_type': 'llm_type', 'confidence': 'llm_confidence', 'reason': 'llm_reason'})
out = bld.merge(_cls[['building_id'] + _KEEP_PLAN + _KEEP_LLM], on='building_id', how='left', validate='one_to_one')
out = out[list(ASSEMBLY_BUILDING_COLS) + _KEEP_PLAN + _KEEP_LLM + ['geometry']]

_required = ['evidence', 'route', 'answered_by', 'mid_labels', 'work_from', 'bosserhof_class', 'llm_confidence']
_nulls = {c: int(out[c].isna().sum()) for c in _required if out[c].isna().any()}
if len(out) != len(bld) or _nulls:
    raise AssertionError(f'the output lost buildings or answers: {len(out):,} rows for {len(bld):,}; nulls {_nulls}')

# --- write -----------------------------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if CLASSIFIED_BUILDINGS_FILE.exists():
    try:
        CLASSIFIED_BUILDINGS_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(f'{CLASSIFIED_BUILDINGS_FILE.name} is locked - close it in QGIS and rerun this cell. '
                           f'Original error: {e}') from None
t0 = time.perf_counter()
for _layer, _frame in (('buildings', out), ('building_pois', pairs)):
    print(f'Writing {len(_frame):,} rows x {len(_frame.columns)} columns -> {CLASSIFIED_BUILDINGS_FILE.name} : {_layer} ...',
          flush=True)
    _frame.to_file(CLASSIFIED_BUILDINGS_FILE, layer=_layer, driver='GPKG')
print(f'  ok  {CLASSIFIED_BUILDINGS_FILE.stat().st_size / 1e6:,.1f} MB  [{time.perf_counter() - t0:,.0f}s]')

# --- read back -----------------------------------------------------------------------
_layers = {l[0]: pyogrio.read_info(CLASSIFIED_BUILDINGS_FILE, layer=l[0])['features']
           for l in pyogrio.list_layers(CLASSIFIED_BUILDINGS_FILE)}
if _layers != {'buildings': len(out), 'building_pois': len(pairs)}:
    raise AssertionError(f'read back {_layers}, expected buildings {len(out):,} and building_pois {len(pairs):,}')
_chk = gpd.read_file(CLASSIFIED_BUILDINGS_FILE, layer='buildings', where="answer_copied = 1", rows=3)
print(f'  ok  read back {_layers}, CRS {_chk.crs}; three buildings with a copied answer:')
print(_chk[['building_id', 'label_en', 'answered_by', 'n_in_signature', 'mid_labels', 'work_from',
            'bosserhof_class', 'llm_confidence']].to_string(index=False))

Writing 39,786 rows x 29 columns -> 06_buildings_classified.gpkg : buildings ...


Writing 41,285 rows x 14 columns -> 06_buildings_classified.gpkg : building_pois ...


  ok  54.6 MB  [3s]
  ok  read back {'buildings': 39786, 'building_pois': 41285}, CRS EPSG:25832; three buildings with a copied answer:
     building_id                           label_en      answered_by  n_in_signature   mid_labels work_from            bosserhof_class llm_confidence
DENIAL01000002A1 Buildings for business or commerce DENIAL0100006qJy              32 work;errands      both business-oriented services         medium
DENIAL01000002D5 Buildings for business or commerce DENIAL0400004UP4             122 work;errands      both                   Services         medium
DENIAL01000002FK Buildings for business or commerce DENIAL0400004UP4             122 work;errands      both                   Services         medium


## Where this leaves us

Sections 1 to 4 are done. Every building of step 04 carries the model's
answer: its own, or — for the class_only buildings — its signature's, marked
`answer_copied`. `work` is on every building, with `work_from` saying whether
the model, the rule or both put it there, and the model's own labels are kept
beside it in `llm_labels`. `06_buildings_classified.gpkg` holds the buildings
and step 04's POI pairs, so it is the one input for the steps after this.

### Next

**06.5, validation.** The answers against the rule baseline in `activities`
and against the annotated Bosserhof set of the previous pipeline.